# Summary

Test calling a deployed agent

In [6]:
import os, sys
import json
import time
from datetime import datetime

from vertexai import agent_engines

import vertexai

# Import libraries from the Agent Development Kit
from google.adk.agents import Agent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.sessions import VertexAiSessionService
from google.genai import types


# Set environment variables
utils_path = "utils/"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
api_configs = ApiAuthentication(client="CCC")

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])


In [2]:
# from search_tools import readVaiSearchResults
# from search_tools import readGoogleSearchResults
#





## Call search tools

In [7]:
q1 = "What are the responsibilities of the board members of a California community college?"
# q1 = "What are LEAP exams"
# q1 = "How many districts are there in the California community college system?"
user_id = "U_123"

# va_results = readVaiSearchResults(query=q1)
#
# gs_results = readGoogleSearchResults(query=q1, user_id=user_id)

from ccc_chatbot_agent import cccChatBot

# Create a chatbot and establish
chatbot = cccChatBot(user_id=user_id)

# Query the agent
cb_results = chatbot.stream_and_parse_query(query=q1)


In [9]:
dir(cb_results)

dir(chatbot)

# chatbot.report_dict["reference_uris"]
#
# ref_uris_md = ["- {}".format(u) for u in chatbot.report_dict["reference_uris"]]
# ref_uris_md

chatbot.report_dict



{'report_title': 'Responsibilities of California Community College Board Members',
 'report_executive_summary': "Board members of California community colleges in California are entrusted with the crucial task of governing individual community college districts. Their responsibilities encompass aligning the district's mission with community needs, ensuring college excellence and sustainability, setting policies, providing guidance, and acting as advocates for the college's interests.",
 'report_body': "### Core Responsibilities\n\n*   **Adapting to Community Needs:** Board members ensure the district's mission aligns with the evolving needs of the community it serves.\n*   **Ensuring Excellence and Sustainability:** A key responsibility is to maintain and enhance the college's academic and operational excellence while ensuring its long-term viability.\n*   **Policy and Guidance:** The Board of Governors sets policies and provides guidance for the 73 districts and 116 colleges within th

In [13]:
dir(chatbot)
len(chatbot.full_context_query)

chatbot.full_context_query


"Use the following search results to synthesize an answer in the context of California community colleges to this user query: What are the responsibilities of the board members of a California community college??  Search results: The California Community Colleges is guided by a process of participatory governance, and the Board of Governors of the California Community Colleges sets policy and provides guidance for the 73 districts and 116 colleges that constitute the system. Board members are appointed by the Governor and formally interact with state and federal officials and other state organizations. The Board of Governors works through a consultation process to ensure representatives from all levels of the system have an opportunity to advise the chancellor on state policy decisions. To obtain information about the California Community Colleges Board of Governors, please contact CCastro@CCCCO.edu. The Board of Governors meet every other month. Full meeting schedule, minutes, and age

In [9]:
cb_results.report_dict.keys()

for key in cb_results.report_dict.keys():
    print(key)
    print(cb_results.report_dict[key])
    print()


report_title
Number of Districts in the California Community College System

report_executive_summary
The California Community Colleges system is structured around local control, comprising 73 community college districts that oversee 116 accredited colleges. These districts are governed by locally elected Boards of Trustees, emphasizing community-based governance within the broader state system.

report_body
The California Community Colleges system consists of 73 community college districts. These districts are responsible for the operation of 116 community colleges throughout California. This structure reflects a commitment to local governance, where locally elected Boards of Trustees work with college presidents to manage individual campuses within their respective districts. The degree of local control in this system can be seen in that 53 of the 73 districts (72%) govern only a single college; only a few districts in major metropolitan areas control more than four colleges.

report

In [10]:
cb_results.report_dict.keys()



dict_keys(['report_title', 'report_executive_summary', 'report_body', 'report_references', 'referece_uris'])

## Combine search contents

In [5]:
va_results.contents
gs_results.contents

contents = va_results.contents + gs_results.contents
contents

long_content_string = " ".join(contents)
print(len(long_content_string))

long_content_string


27578


"Hard Copy The board chair leads the board of trustees and facilitates board processes. He or she plays an important role in ensuring that the board effectively governs the institution and that trustees work together well. The chair is often perceived as the major spokesperson for the board. She or he is usually the primary point of contact with the chief executive of the district. This handbook provides important best practices for the board chair to follow. Updated January 2024. Phone: 916-444-8641 Fax: 916-444-2954 cclc@ccleague.org 2017 O Street Sacramento, CA 95811 Sign up for our weekly news and event announcements. Δ Social © 2025 Community College League of California. All rights reserved. The California Community College Trustees (CCCT) Board serves an essential purpose within the Community College League of California. Meeting five times a year, the twenty-two member board provides leadership and direction to ensure a strong voice for governing board members. Nominations for 

## Call the synthesis agents

In [6]:
query = ("Use the following search results to synthesize an answer to this user query: {}?  "
         "Search results: {}.").format(q1,
                                       long_content_string)

query

"Use the following search results to synthesize an answer to this user query: What are the responsibilities of the board members of a California community college??  Search results: Hard Copy The board chair leads the board of trustees and facilitates board processes. He or she plays an important role in ensuring that the board effectively governs the institution and that trustees work together well. The chair is often perceived as the major spokesperson for the board. She or he is usually the primary point of contact with the chief executive of the district. This handbook provides important best practices for the board chair to follow. Updated January 2024. Phone: 916-444-8641 Fax: 916-444-2954 cclc@ccleague.org 2017 O Street Sacramento, CA 95811 Sign up for our weekly news and event announcements. Δ Social © 2025 Community College League of California. All rights reserved. The California Community College Trustees (CCCT) Board serves an essential purpose within the Community College 

In [7]:

# Retreive agent
# resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/8712609303334223872"
# resource_name = "projects/eternal-bongo-435614-b9/locations/us-central1/reasoningEngines/2582084310576136192"
# resource_name ="projects/1062597788108/locations/us-central1/reasoningEngines/5119862700599410688"
# resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/7268079722855137280"
resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/3177122411342462976"
agent_engine = agent_engines.get(resource_name)

# Establish session
session = agent_engine.create_session(user_id=user_id)

# Get agent response
result = agent_engine.stream_query(message=query,
                                   session_id=session["id"],
                                   user_id=user_id)


# Put results into a dictionary for later access
events = []
for event in result:
    events.append(event)

## Provide a synthesized answer

In [8]:
events

for event in events:
    if type(event) == dict and "content" in event.keys():
        print(event["content"]["parts"][0]["text"])

```json
{
  "report_title": "Responsibilities of California Community College Board Members",
  "report_executive_summary": "Board members of California Community Colleges in California have a wide range of responsibilities focused on effective governance, policy-setting, and ensuring student success. These responsibilities include setting the educational priorities of the district, ensuring college excellence and sustainability, and providing guidance for the community college system. Board members are expected to adhere to fiduciary duties, including the duty of care, loyalty, and obedience, and establish standards for local district consultation processes.",
  "report_body": "### Core Responsibilities of Board Members\n\n*   **Policy and Guidance:** Setting policies and providing guidance for the 73 districts and 116 colleges within the California Community Colleges system.\n*   **Strategic Focus:** Focusing on key issues related to fulfilling the district's mission and shifting fro

## Show links and organizations

In [17]:
print("Here are additional resources\n")
print("Organizations who have helpful content")
print(va_results.organizations)


print("\nUseful links")
print(va_results.uris + gs_results.uris)

print()

Here are additional resources

Organizations who have helpful content
[{'name': 'California Community Colleges', 'role': 'Informational site for all California community colleges.', 'uri': 'https://www.cccco.edu'}, {'role': 'The Community College League of California is made up of a small, but mighty team that is dedicated to serving our members – California’s 73 community college districts.', 'name': 'Community College League of California', 'uri': 'https://www.ccleague.org'}]

Useful links
['https://www.cccco.edu/About-Us/Board-of-Governors', 'https://www.ccleague.org/archive/newly-elected-trustee-here-are-your-next-steps/', 'https://www.ccleague.org/news/7507/', 'https://www.cccco.edu/About-Us#page-banner', 'https://www.ccleague.org/about-us/our-team/', 'https://www.cccco.edu/About-Us', 'https://www.ccleague.org/product/board-chair-handbook-hard-copy/', 'https://www.cccco.edu/About-Us/Board-of-Governors#page-banner', 'https://www.ccleague.org/trustees/', 'https://www.ccleague.org/on

####################

## Retrieve a deployed agent

In [2]:
dir(agent_engines)

# get a list of agents
for agent in agent_engines.list():
    print(agent.resource_name)

# Delete if needed
# for agent in agent_engines.list():
#     agent_engines.delete(resource_name=agent.resource_name,
#                          force=True)


projects/1062597788108/locations/us-central1/reasoningEngines/2582084310576136192
projects/1062597788108/locations/us-central1/reasoningEngines/6068151897137610752
projects/1062597788108/locations/us-central1/reasoningEngines/5673523979789271040


In [3]:
# os.getenv("AGENT_ENGINE_ID")

In [9]:
vertexai_resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/1949187825442226176"
# search_resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/6068151897137610752"

resource_name = vertexai_resource_name


agent_engine2 = agent_engines.get(resource_name)
print("Testing a local deployment of agent: {}".format(agent_engine2.name))
start_time = datetime.now()
print("Time: {}".format(start_time.strftime("%b %-d, %Y, %-H:%M:%S %p")))


session = agent_engine2.create_session(user_id="u_123")
print("Session details: {}".format(session))

q1 = "What are the responsibilities of the board members of a California community college?"
test_result = agent_engine2.stream_query(message=q1,
                                        session_id=session["id"],
                                        user_id="U_123")
events = []
for event in test_result:
    events.append(event)

    print("Event Author: {}".format(event["author"]))
    print("Text: {} ...".format(event['content']["parts"][0]["text"][:750]))
    print()


Testing a local deployment of agent: 1949187825442226176
Time: Jun 24, 2025, 16:16:14 PM
Session details: {'lastUpdateTime': 1750799775.993789, 'appName': '1949187825442226176', 'id': '449511689516220416', 'state': {}, 'events': [], 'userId': 'u_123'}


In [21]:

# Retreive an ADK agent
# agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))




# dir(agent_engine)
# Opt 1: Create a session using agent_engine
session = agent_engine.create_session(user_id="u_123")



#### Option 2
# session_service = VertexAiSessionService(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                          location=os.environ["GOOGLE_CLOUD_LOCATION"])
#
# # type(agent_engine)
# session_service_active1 = agent_engine.create_session(user_id="u_123")


In [11]:
print(session)
session['id']

{'userId': 'u_123', 'appName': '9020261452878970880', 'id': '8902275608881397760', 'events': [], 'state': {}, 'lastUpdateTime': 1750173511.784099}


'8902275608881397760'

In [9]:
type(agent_engine)
# dir(agent_engines)
# agent_engine.operation_schemas()
# dir(session)


vertexai.agent_engines._agent_engines.AgentEngine

In [6]:
agent_engine

resource name: projects/1062597788108/locations/us-central1/reasoningEngines/9020261452878970880

In [8]:
# # from google.adk.sessions import VertexAiSessionService
# #
# app_name=os.getenv("AGENT_ENGINE_ID")
# user_id="u_123"
#
# # Create the ADK runner with VertexAiSessionService
# session_service = VertexAiSessionService(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                          location=os.environ["GOOGLE_CLOUD_LOCATION"])
# runner = Runner(
#     # agent=root_agent,
#     agent=agent_engine,
#     app_name=app_name,
#     session_service=session_service)
#
#
# # Helper method to send query to the runner
# def call_agent(query, session_id, user_id):
#   content = types.Content(role='user', parts=[types.Part(text=query)])
#   events = runner.run(
#       user_id=user_id, session_id=session_id, new_message=content)
#
#   for event in events:
#       if event.is_final_response():
#           final_response = event.content.parts[0].text
#           print("Agent Response: ", final_response)
#
# q1 = "What are LEAP exams?"
# # q2 = "What are the responsibilities of the board members of a California community college?"
#
# test_result2 = call_agent(query=q1, session_id=user_id="U_123")

In [9]:
# type(agent_engine)
# dir(session_service)

# session.keys()

## Test the agent with one query

In [22]:
q1 = "What are LEAP exams?"
q1 = "What are the responsibilities of the board members of a California community college?"

test_result = agent_engine.stream_query(message=q1,
                                        session_id=session["id"],
                                        user_id="U_123")


# test_result = agent_engine.async_stream_query(message=q1, user_id="U_123")

events = []
for event in test_result:
    # print(event)
    events.append(event)



In [23]:
for event in events:
    print(type(event))
    print(event.keys())
    print(event['content'].keys())
    # print(event['content']['parts'])
    # print(event['grounding_metadata'])
    print(event['author'])
    print(event['actions'])
    print()




<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'usage_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
search_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}



In [31]:
event['content']["parts"][0]["text"][:750]

"The responsibilities of board members of a California community college are multifaceted, revolving around governance, policy setting, and ensuring the well-being of the college district. Here's a breakdown of their key duties:\n\n**1. Governance and Policy:**\n\n*   **Policy Setting:** Board members are responsible for establishing policies that define the institutional mission and set standards for college operations, ensuring these policies align with legal and ethical guidelines.\n*   **Strategic Direction:** They set the college's strategic direction, focusing on the future learning needs of their communities.\n*   **Monitoring Performance:** They monitor the institution's performance, holding the colleges accountable for achieving student s"

In [7]:
for event in events:
    if event["author"] == "rag_search_agent":
        print(event["author"])
        e1 = event
        for entry in event["content"]["parts"]:
            try:
                print(entry["text"])
            except:
                pass

In [8]:
events

[]

In [ ]:
e1

In [ ]:
e1.keys()

for key in e1.keys():
    # print(key)
    # print(e1[key])
    # print()
    if key == "grounding_metadata":
        print(e1[key].keys())
        print()
        for gc in e1[key]:
            # print(gc)
            # print(e1[key][gc])
            # print()
            #
            # get urls

            # get metadata
            if gc == "grounding_chunks":
                for gcli in e1[key]["grounding_chunks"]:
                    # print(gcli["retrieved_context"])

                    for gclikey in gcli["retrieved_context"].keys():

                        for gcl4key in gcli["retrieved_context"][gclikey].keys():
                            print(gcl4key)
                            print(type(gcli["retrieved_context"][gclikey][gcl4key]))
                            print("====")

                        # print(gclikey)
                        # print(type(gcli["retrieved_context"][gclikey]))
                        # print(gcli["retrieved_context"][gclikey])
                        #
                        # print("====")

    #


In [ ]:
import re

finds_list = []
for item in e1["grounding_metadata"]["grounding_chunks"]:
    # print(type(item["retrieved_context"]["rag_chunk"]["text"]))
    # print(item["retrieved_context"]["rag_chunk"]["text"])
    # print("===")

    pats = {"title": r"structData title (.*)", "page_url": r"structData page_url (.*)"}


    for patkey in pats:

        patfinds = re.findall(pats[patkey], item["retrieved_context"]["rag_chunk"]["text"])
        if patfinds:
            for patfind in patfinds:
                finds_list.append({patkey: patfind})



In [ ]:
e1["grounding_metadata"]["grounding_chunks"][3]

In [ ]:
finds_list

# t2 = finds[1]
# dir(t2)
#
# t2.string

# find_between(item["retrieved_context"]["rag_chunk"]["text"], "StructData", "StructData")

In [ ]:
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions
import functions_framework
import json

# Your Project, Location, and Data Store ID
PROJECT_ID = "your-gcp-project-id"
LOCATION = "global"  # Or your specific location
DATA_STORE_ID = "your-data-store-id"

@functions_framework.http
def search_with_metadata(request):
    """
    Receives a query from a Vertex Agent and returns a response
    with structured metadata.
    """
    request_json = request.get_json(silent=True)
    query = request_json['sessionInfo']['parameters']['query'] # Adjust based on how agent sends params

    if not query:
        return json.dumps({"fulfillment_response": {"messages": [{"text": {"text": ["Please provide a search query."]}}]}})

    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
    )
    serving_config = client.serving_config_path(
        project=PROJECT_ID, location=LOCATION,
        data_store=DATA_STORE_ID, serving_config="default_config"
    )

    search_request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=3 # Get top 3 results
    )

    response = client.search(search_request)

    # Format the response for the agent
    final_response = "Here are the top results for your query:\n\n"
    for i, result in enumerate(response.results):
        doc_data = result.document.struct_data
        final_response += f"--- Result {i+1} ---\n"
        final_response += f"Snippet: ...{result.document.derived_struct_data['snippets'][0]['snippet']}...\n"
        final_response += f"Author: {doc_data.get('author', 'N/A')}\n"
        final_response += f"Source URL: {doc_data.get('source_url', 'N/A')}\n"
        final_response += f"Document ID: {result.document.id}\n\n"


    # Create the response JSON for Vertex AI Agent Builder
    response_json = {
        "fulfillment_response": {
            "messages": [
                {
                    "text": {
                        "text": [final_response]
                    }
                }
            ]
        }
    }

    return json.dumps(response_json)

In [ ]:
# agent_engines.get("4226237373903536128")
# dir(agent_engines)

# dir(agent_engine)
#
# test_session = agent_engine.get_session(user_id="U_123", session_id="4226237373903536128")
#
# dir(test_session)
#
# dir(test_session.items())



In [ ]:
# for event in events:
#     if event["author"] == "synthesis_agent":
#         for entry in event["content"]["parts"]:
#             print(entry["text"])
#
for event in events:
#     # if event["author"] == "synthesis_agent":
    print(event["author"])
    for entry in event["content"]["parts"]:
        try:
            print(entry["text"])
        except:
            pass
#
    print()
    print()


# events


## Test the agent with multiple queries in a session

In [ ]:
def pretty_print_event(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        print(f"[{event.get('author', 'unknown')}]: {event}")
        return

    author = event.get("author", "unknown")
    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            # Truncate long text to 200 characters
            if len(text) > 200:
                text = text[:197] + "..."
            print(f"[{author}]: {text}")
        elif "functionCall" in part:
            func_call = part["functionCall"]
            print(f"[{author}]: Function call: {func_call.get('name', 'unknown')}")
            # Truncate args if too long
            args = json.dumps(func_call.get("args", {}))
            if len(args) > 100:
                args = args[:97] + "..."
            print(f"  Args: {args}")
        elif "functionResponse" in part:
            func_response = part["functionResponse"]
            print(f"[{author}]: Function response: {func_response.get('name', 'unknown')}")
            # Truncate response if too long
            response = json.dumps(func_response.get("response", {}))
            if len(response) > 100:
                response = response[:97] + "..."
            print(f"  Response: {response}")

agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
print(f"Agent ID: Id='{agent_engine.resource_name}'")

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)
print(f"Session created: User='{user_id}', Session='{session['id']}'")

# Create some queries
queries = [
    "Hi, how are you?",
    "What are LEAP exams?",
    "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
]

# Look for events for each query
for query in queries:
    print(f"\n[user]: {query}")
    for event in agent_engine.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=query,
    ):
        print("We got here by finding an event")
        pretty_print_event(event)
        print(session.get(session["id"]))


## Look at the sessions

In [ ]:
agent_engine.list_sessions(user_id=user_id)

## Create - This stuff doesn't work with the deployed agent but does work if the deployed agent is called directly from code

In [ ]:

def send_query_to_agent(agent, query, user_id):
    """Sends a query to the specified agent and prints the response.

    Args:
        agent: The agent to send the query to.
        query: The query to send to the agent.

    Returns:
        A tuple containing the elapsed time (in milliseconds) and the final response from the agent.
    """

    # Create a new session - if you want to keep the history of interruction you need to move the
    # creation of the session outside of this function. Here we create a new session per query
    session = session_service.create_session(app_name=AGENT_APP_NAME,
                                             user_id=user_id,)
    # Create a content object representing the user's query
    print('\nUser Query: ', query)
    content = types.Content(role='user', parts=[types.Part(text=query)])

    # Start a timer to measure the response time
    start_time = time.time()

    # Create a runner object to manage the interaction with the agent
    runner = Runner(app_name=AGENT_APP_NAME, agent=agent, artifact_service=artifact_service, session_service=session_service)

    # Run the interaction with the agent and get a stream of events
    events = runner.run(user_id=user_id, session_id=session.id, new_message=content)

    final_response = None
    elapsed_time_ms = 0.0

    # Loop through the events returned by the runner
    for _, event in enumerate(events):

        is_final_response = event.is_final_response()

        if not event.content:
             continue

        if is_final_response:
            end_time = time.time()
            elapsed_time_ms = round((end_time - start_time) * 1000, 3)

            print("-----------------------------")
            print('>>> Inside final response <<<')
            print("-----------------------------")
            final_response = event.content.parts[0].text # Get the final response from the agent
            print(f'Agent: {event.author}')
            print(f'Response time: {elapsed_time_ms} ms\n')
            print(f'Final Response:\n{final_response}')
            print("----------------------------------------------------------\n")

    return elapsed_time_ms, final_response



In [ ]:
MODEL='gemini-2.0-flash-001'

# # Create a basic agent with instructions amd greeting only
# basic_agent = Agent(model=MODEL,
#     name="agent_basic",
#     description="This agent responds to inquiries about its creation by stating it was built using the Google Agent Framework.",
#     instruction="If they ask you how you were created, tell them you were created with the Google Agent Framework.",
#     generate_content_config=types.GenerateContentConfig(temperature=0.2),
# )

############## Import the agent code from the python file used to create the deployed agent
rag_path = "ccc_chatbot/"
sys.path.insert(0, rag_path)
print(os.listdir(rag_path))
from agent import root_agent

basic_agent = root_agent

# Get the agent
# basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(basic_agent, "How are you today", user_id)


In [ ]:
from google.adk.agents import SequentialAgent
from google.adk.agents import LlmAgent

basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

code_pipeline_agent = SequentialAgent(
    name="run_other_agents",
    # sub_agents=[basic_agent],
    tools=[basic_agent],
    description="Wrapper for the basic agent",
)

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(code_pipeline_agent, "How are you today", user_id)